# **任务17 残差网络 - 多层感知机 | Residual Network (ResNet) - MLP**

残差网络是对传统深度神经网络的改良，而非是一种新的层结构，我们可以在任何神经网络层中应用它。

残差网络是用于解决当模型层数加深，非线性激活函数增多，导致反向传播不能有效传递到较深网络。采用 残差快输入+残差块输出 作为下一层输入的方式，将上一层的特征构建连接，对于模型来说学习 **x1 对 x1+x0** 的影响 比 **x1 对 x0** 的影响更简单，这种方式更有利于深层神经网络的学习。

## 1. 定义残差快

In [7]:
import torch
import torch.nn as nn

class ResBlock(nn.Module):
    def __init__(self, input_dim, output_dim, hidden_dim=None):
        super().__init__()
        if not hidden_dim:
            hidden_dim = input_dim

        self.layer = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
        )
        self.output_layer = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        out = self.layer(x)
        out = out + x       # 残差连接
        out = self.output_layer(out)
        return out

## 2. **定义模型**

使用线性层残差快直接替换MLP的线性层即可。

In [8]:
class MainModel(nn.Module):
    def __init__(self, input_dim, output_dim, hidden_dim):
        super().__init__()
        self.layer = nn.Sequential(
            ResBlock(input_dim, hidden_dim),
            nn.ReLU(),
            ResBlock(hidden_dim, hidden_dim),
            nn.ReLU(),
            ResBlock(hidden_dim, hidden_dim)
        )
        self.output_layer = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        out = self.layer(x)
        out = self.output_layer(out)
        return out

## 3. **模拟前向传播**

In [9]:
batch_size = 8
input_tensor = torch.rand((batch_size, 3))

model = MainModel(3, 5, 16)

# 前向传播过程
output_tensor = model(input_tensor)

print('input_tensor:', input_tensor.shape)
print('output_tensor:', output_tensor.shape)


input_tensor: torch.Size([8, 3])
output_tensor: torch.Size([8, 5])


卷积残差块及残差网络总结见下一任务